### 명확한 인스트럭션
- bad
    - 감정을 찾아줘
- good
    - 다음 영화 리뷰의 감정을 긍정/부정/중립 중 하나로 분류하세요 응답은 감정 단어 하나만 출력하세요

### 일관된 포멧
- 리뷰1 : ...
- 감정 : 긍정

- 리뷰2 : ...
- 감정 : 부정

- 리뷰 : ...
- 감정 : 긍정

- 리뷰 : ...
- 감정 : 부정

### 다양한 예시 : 균형잡히게
- 예시 1 : "좋아" ->긍정
- 예시 1 : "최고야" ->긍정
- 예시 1 : "훌륭해" ->긍정

- 예시 1 : "좋아" ->긍정
- 예시 1 : "싫어" ->부정
- 예시 1 : "그럭저럭" ->중립

### 예시의 순서효과
- 긍정1, 긍정2, 부정, 중립 -> 모델이 긍정을 과도하게 예측 과적합 상태

- 긍정,부정,중립 -> 균형있게 학습

### 추천
- 클래스별 균형
- 어려운 경우부터 시작(모델이 주의집중)
- 같은 클래스 반복 피하기
- 무작위 섞기보다 신중하게 배열

### 전략
- 다양성 극대화
    - 선택기준:
        - 각 클래스를 대표하는 명확한 사례
        - 경계 사례 포함(모호한 케이스)
        - 도메일 특화 어휘 포함
- 어려움 레벨 고려
    - Easy : '이 영화 최고야! (명확한 긍정)
    - Midium : '그런대로 나쁘지 않네'(부정 + 약한표현)
    - Hard : .... (도메인특화, 뉘양스 중요)

    - 프롬프트 : Easy->Medium->Hard 순서


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations

In [2]:
# 법률 문서 조항 데이터
legal_data = [
    {
        "text": "본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일까지이다.",
        "category": "기간",
        "difficulty": "easy"
    },
    {
        "text": "계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 해제할 수 있다.",
        "category": "위약금",
        "difficulty": "medium"
    },
    {
        "text": "갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다.",
        "category": "책임제한",
        "difficulty": "medium"
    },
    {
        "text": "본 계약에 명시되지 않은 사항에 대해서는 관계 법령을 준용한다.",
        "category": "준거법",
        "difficulty": "hard"
    },
    {
        "text": "이용자는 언제든지 7일 이전 통지로 본 계약을 해지할 수 있다.",
        "category": "해지",
        "difficulty": "easy"
    },
    {
        "text": "당사자 간의 분쟁이 발생할 경우 서울중앙지방법원을 관할 법원으로 한다.",
        "category": "관할법원",
        "difficulty": "hard"
    },
]

test_data = [
    {"text": "본 서비스는 2024년 1월부터 시작된다.", "category": "기간"},
    {"text": "계약 위반 시 연 10%의 위약금을 청구할 수 있다.", "category": "위약금"},
    {"text": "본사는 서비스 중단에 대해 책임지지 않는다.", "category": "책임제한"},
    {"text": "분쟁은 중재법에 따라 처리된다.", "category": "준거법"},
]

print(f"훈련 데이터: {len(legal_data)}개")
for i, d in enumerate(legal_data, 1):
    print(f"  {i}. [{d['category']:8}] {d['text'][:40]}...")

print(f"\n테스트 데이터: {len(test_data)}개")
for i, d in enumerate(test_data, 1):
    print(f"  {i}. {d['text'][:50]}...")

훈련 데이터: 6개
  1. [기간      ] 본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일...
  2. [위약금     ] 계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 ...
  3. [책임제한    ] 갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다...
  4. [준거법     ] 본 계약에 명시되지 않은 사항에 대해서는 관계 법령을 준용한다....
  5. [해지      ] 이용자는 언제든지 7일 이전 통지로 본 계약을 해지할 수 있다....
  6. [관할법원    ] 당사자 간의 분쟁이 발생할 경우 서울중앙지방법원을 관할 법원으로 한다....

테스트 데이터: 4개
  1. 본 서비스는 2024년 1월부터 시작된다....
  2. 계약 위반 시 연 10%의 위약금을 청구할 수 있다....
  3. 본사는 서비스 중단에 대해 책임지지 않는다....
  4. 분쟁은 중재법에 따라 처리된다....


In [ ]:
class PromptOptimizer:
    """프롬프트 최적화 클래스"""
    
    def __init__(self):
        self.category_map = {
            "기간": ["시간", "기간", "날짜", "년도", "일자"],
            "위약금": ["위약금", "손해배상", "페널티", "과태료", "위반"],
            "책임제한": ["책임", "담당", "책임제한", "면책", "책임지지"],
            "준거법": ["법령", "준거법", "법률", "규정", "법"],
            "해지": ["해지", "종료", "종료", "폐기", "중단"],
            "관할법원": ["법원", "중재", "소송", "분쟁", "관할"],
        }
    
    def classify_simple(self, text):
        """키워드 기반 분류"""
        text_lower = text.lower()
        scores = {}
        
        for category, keywords in self.category_map.items():
            score = sum(1 for kw in keywords if kw in text_lower)
            scores[category] = score
        
        if max(scores.values()) == 0:
            return "기타"
        
        return max(scores, key=scores.get)

optimizer = PromptOptimizer()
print("프롬프트 최적화 클래스 초기화 완료")

In [3]:
def create_prompt(examples, order="original"):
    """프롬프트 생성"""
    
    if order == "random":
        examples = list(examples)
        np.random.shuffle(examples)
    elif order == "difficulty":
        # 어려움 순서: easy → medium → hard
        difficulty_order = {"easy": 0, "medium": 1, "hard": 2}
        examples = sorted(examples, key=lambda x: difficulty_order.get(x.get("difficulty", "easy"), 0))
    elif order == "balanced":
        # 각 카테고리가 고르게 분포
        by_category = {}
        for ex in examples:
            cat = ex["category"]
            if cat not in by_category:
                by_category[cat] = []
            by_category[cat].append(ex)
        
        examples = []
        max_len = max(len(v) for v in by_category.values())
        for i in range(max_len):
            for cat in by_category:
                if i < len(by_category[cat]):
                    examples.append(by_category[cat][i])
    
    prompt = """다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간, 위약금, 책임제한, 준거법, 해지, 관할법원
응답 형식: 카테고리명만 출력하세요.

"""
    
    for ex in examples:
        prompt += f"조항: {ex['text']}\n"
        prompt += f"카테고리: {ex['category']}\n\n"
    
    return prompt, examples

# 4가지 순서로 프롬프트 생성
orders = ["original", "random", "difficulty", "balanced"]

for order in orders:
    prompt, ex_order = create_prompt(legal_data[:3], order=order)
    print(f"\n{'='*60}")
    print(f"[순서: {order.upper()}]")
    print(f"{'='*60}")
    print(prompt[:300] + "...")
    print(f"예시 순서: {[ex['category'] for ex in ex_order]}")


[순서: ORIGINAL]
다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간, 위약금, 책임제한, 준거법, 해지, 관할법원
응답 형식: 카테고리명만 출력하세요.

조항: 본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일까지이다.
카테고리: 기간

조항: 계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 해제할 수 있다.
카테고리: 위약금

조항: 갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다.
카테고리: 책임제한

...
예시 순서: ['기간', '위약금', '책임제한']

[순서: RANDOM]
다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간, 위약금, 책임제한, 준거법, 해지, 관할법원
응답 형식: 카테고리명만 출력하세요.

조항: 본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일까지이다.
카테고리: 기간

조항: 갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다.
카테고리: 책임제한

조항: 계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 해제할 수 있다.
카테고리: 위약금

...
예시 순서: ['기간', '책임제한', '위약금']

[순서: DIFFICULTY]
다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간, 위약금, 책임제한, 준거법, 해지, 관할법원
응답 형식: 카테고리명만 출력하세요.

조항: 본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일까지이다.
카테고리: 기간

조항: 계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 해제할 수 있다.
카테고리: 위약금

조항: 갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다.
카테고리: 책임제한

...
예시 순서: ['기간', '위약금', '책임제한']

[순서: BALANCED]
다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간,